# Project 4 | NHS Data Engineering Portfolio | Yusuf Ismail
## NHS Cancer Waiting Times — Data Exploration

**Notebook 00 of 5** — I use this notebook to document the raw data before I build anything.
Nothing here is transformed or saved. I am only looking, confirming, and recording what
the NHS England Combined CSV files actually contain, so that anyone reading my repo
understands the dataset before they reach my Bronze ingestion code.

**Source:** NHS England Cancer Waiting Times — Monthly Combined CSV (Provider + Commissioner)
https://www.england.nhs.uk/statistics/statistical-work-areas/cancer-waiting-times/

**Files I am working with (3 financial years, all Final):**
- `2022-23-Apr-Mar-Monthly-Combined-CSV-Final.csv`
- `2023-24-Apr-Mar-Monthly-Combined-CSV-Final.csv`
- `2024-25-Apr-Mar-Monthly-Combined-CSV-Final.csv`

**The three standards in this data:**
- **FDS** — Faster Diagnosis Standard (28 days, target 75% rising to 80% by 2026)
- **31D** — 31-day standard, decision-to-treat to first treatment (target 96%)
- **62D** — 62-day standard, urgent referral to first treatment (target 85%, interim 70%)

**Analysis scope decision:** I focus my Silver and Gold work on October 2023 onwards.
The NHS changed the cancer standards framework in October 2023, so data before that date
is not directly comparable for the 31-day and 62-day standards. 18 months of clean,
comparable data (Oct 2023 – Mar 2025) tells a stronger story than 36 months with a break in it.

In [2]:
# I import only what I need for exploration — pandas for the data,
# pathlib for clean, OS-independent file paths (I am on macOS but this keeps it portable).
import pandas as pd
from pathlib import Path

# I set my display options so wide NHS tables don't get truncated when I inspect them.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# I define my project paths once, here, so every later cell refers back to these.
# RAW_DIR is where my three downloaded CSVs live. I never write to it — it stays read-only in spirit.
PROJECT_DIR = Path.cwd().parent          # notebooks/ sits inside the project, so parent = project root
RAW_DIR = PROJECT_DIR / "data" / "raw"

# I list the three files explicitly rather than globbing, so I know exactly what I am loading
# and the order is predictable (oldest financial year first).
RAW_FILES = {
    "2022-23": RAW_DIR / "2022-23-Apr-Mar-Monthly-Combined-CSV-Final.csv",
    "2023-24": RAW_DIR / "2023-24-Apr-Mar-Monthly-Combined-CSV-Final.csv",
    "2024-25": RAW_DIR / "2024-25-Apr-Mar-Monthly-Combined-CSV-Final.csv",
}

# I confirm the files are actually where I expect before going any further.
print("Project root:", PROJECT_DIR)
print("Raw data folder:", RAW_DIR)
print()
for year, path in RAW_FILES.items():
    exists = path.exists()
    size_mb = path.stat().st_size / 1_000_000 if exists else 0
    print(f"  {year}:  {'FOUND' if exists else 'MISSING'}  ({size_mb:.1f} MB)  {path.name}")

Project root: /Users/yusufismail/Documents/cancer_waiting_time
Raw data folder: /Users/yusufismail/Documents/cancer_waiting_time/data/raw

  2022-23:  FOUND  (56.1 MB)  2022-23-Apr-Mar-Monthly-Combined-CSV-Final.csv
  2023-24:  FOUND  (60.1 MB)  2023-24-Apr-Mar-Monthly-Combined-CSV-Final.csv
  2024-25:  FOUND  (64.4 MB)  2024-25-Apr-Mar-Monthly-Combined-CSV-Final.csv


## Step 1 — Load one file and inspect its structure

I start with the 2024-25 file because it sits entirely within the new (post-October 2023)
framework, so it shows me the schema I will actually be analysing. I check the shape,
the columns, and the data types before I look at any values.

In [11]:
# I load the most recent financial year first. low_memory=False stops pandas from
# guessing column types chunk-by-chunk, which avoids mixed-type warnings on the
# wide band columns that are mostly empty for some standards.
df_2425 = pd.read_csv(RAW_FILES["2024-25"], low_memory=False)

# I check the size first — rows and columns — so I know the scale I am dealing with.
print("Shape (rows, columns):", df_2425.shape)
print()

# I print every column name with its position and dtype. This is the single most
# important thing to record: the exact schema NHS England gives me, in their wording.
print("Columns and types:")
print(df_2425.dtypes)

Shape (rows, columns): (324758, 30)

Columns and types:
Period                      object
Basis                       object
Org_Code                    object
Parent_Org                  object
Org_Name                    object
Standard_or_Item            object
Cancer_Type                 object
Referral_Route_or_Stage     object
Treatment_Modality          object
Total                      float64
Within                     float64
After                      float64
Performance                float64
Within_14_days             float64
In_15_to_28_days           float64
In_29_to_42_days           float64
In_43_to_62_days           float64
After_62_days              float64
Within_31_days             float64
In_32_to_38_days           float64
In_39_to_48_days           float64
In_49_to_62_days           float64
In_63_to_76_days           float64
In_77_to_90_days           float64
In_91_to_104_days          float64
After_104_days             float64
In_15_to_16_days           float64

## Step 2 — Understand the four column blocks

The 30 columns fall into four logical blocks. Recording this is the whole point of
this notebook — once I understand the structure, the Bronze and Silver code writes itself.

**Block 1 — Identity & time (cols 1–5):** `Period`, `Basis`, `Org_Code`, `Parent_Org`, `Org_Name`
Tells me *who* and *when*. `Basis` (Provider / Commissioner) is critical — I must filter
to one basis or I double-count every patient.

**Block 2 — Dimensions (cols 6–9):** `Standard_or_Item`, `Cancer_Type`, `Referral_Route_or_Stage`, `Treatment_Modality`
Tells me *what slice* each row is. A row only has meaning once I know all four. This is the
long-thin design: one row per combination of dimensions.

**Block 3 — Headline measures (cols 10–13):** `Total`, `Within`, `After`, `Performance`
The numbers I aggregate. Total = patients, Within = met standard, After = breached,
Performance = pre-calculated proportion (I will recalculate my own to be safe).

**Block 4 — Waiting-time bands (cols 14–30):** the depth RTT and A&E never had.
These tell me *how late*, not just late/not-late. Different bands populate for different
standards. I verified the bands reconcile exactly to Total, so they are trustworthy.

**Important finding — this file holds FIVE items, not three.** Beyond the three compliance
standards (FDS, 31D, 62D), there are two referral-activity items:
`Urgent Suspected Cancer referral` (USC) and `Urgent Breast Symptomatic referral` (UBS).
These measure how fast a patient gets their FIRST appointment after referral — the old
"two-week-wait" style clock — so they use their own finer day-bands
(`In_15_to_16_days`, `In_17_to_21_days`, `In_22_to_28_days`, `After_28_days`).
I verified these reconcile exactly to Total. Nothing gets dropped — every band belongs to
some item. My three articles focus on FDS/31D/62D, but I keep USC/UBS in Bronze for completeness.

In [14]:
# I want to prove to myself (and anyone reading this) exactly which waiting-time band
# columns are actually populated for each standard. I never trust column names at face
# value — NHS England reuses band columns across standards in this flat layout.

band_cols = [
    'Within_14_days','In_15_to_28_days','In_29_to_42_days','In_43_to_62_days','After_62_days',
    'Within_31_days','In_32_to_38_days','In_39_to_48_days','In_49_to_62_days','In_63_to_76_days',
    'In_77_to_90_days','In_91_to_104_days','After_104_days',
    'In_15_to_16_days','In_17_to_21_days','In_22_to_28_days','After_28_days'
]

# For each standard, I list which band columns carry any data at all.
for std in ['FDS', '31D', '62D']:
    sub = df_2425[df_2425['Standard_or_Item'] == std]
    populated = [c for c in band_cols if sub[c].notna().any()]
    print(f"{std}  ({len(sub):,} rows) — {len(populated)} populated band columns:")
    for c in populated:
        print(f"    • {c}")
    print()

# I confirm the four legacy columns are empty across the whole file, so I can drop them safely.
legacy = ['In_15_to_16_days','In_17_to_21_days','In_22_to_28_days','After_28_days']
print("Legacy columns (expect 0 non-null each):")
for c in legacy:
    print(f"    {c}: {df_2425[c].notna().sum()} non-null")

FDS  (66,837 rows) — 5 populated band columns:
    • Within_14_days
    • In_15_to_28_days
    • In_29_to_42_days
    • In_43_to_62_days
    • After_62_days

31D  (101,671 rows) — 5 populated band columns:
    • After_62_days
    • Within_31_days
    • In_32_to_38_days
    • In_39_to_48_days
    • In_49_to_62_days

62D  (125,190 rows) — 9 populated band columns:
    • After_62_days
    • Within_31_days
    • In_32_to_38_days
    • In_39_to_48_days
    • In_49_to_62_days
    • In_63_to_76_days
    • In_77_to_90_days
    • In_91_to_104_days
    • After_104_days

Legacy columns (expect 0 non-null each):
    In_15_to_16_days: 31060 non-null
    In_17_to_21_days: 31060 non-null
    In_22_to_28_days: 31060 non-null
    After_28_days: 31060 non-null


In [17]:
# I now map ALL FIVE items in this file to their band columns, not just the three standards.
# The two referral items (USC, UBS) use a finer day-band set because they measure
# referral-to-first-appointment on a tighter clock than the treatment standards.

all_items = df_2425['Standard_or_Item'].unique()
print("All items in this file:")
for item in all_items:
    n = (df_2425['Standard_or_Item'] == item).sum()
    print(f"    • {item}  ({n:,} rows)")
print()

# For each item, I show which band columns carry data — the definitive structure map.
all_band_cols = [
    'Within_14_days','In_15_to_28_days','In_29_to_42_days','In_43_to_62_days','After_62_days',
    'Within_31_days','In_32_to_38_days','In_39_to_48_days','In_49_to_62_days','In_63_to_76_days',
    'In_77_to_90_days','In_91_to_104_days','After_104_days',
    'In_15_to_16_days','In_17_to_21_days','In_22_to_28_days','After_28_days'
]

for item in all_items:
    sub = df_2425[df_2425['Standard_or_Item'] == item]
    populated = [c for c in all_band_cols if sub[c].notna().any()]
    print(f"{item}:")
    print(f"    {populated}")
    print()

All items in this file:
    • FDS  (66,837 rows)
    • 31D  (101,671 rows)
    • 62D  (125,190 rows)
    • Urgent Suspected Cancer referral  (29,192 rows)
    • Urgent Breast Symptomatic referral  (1,868 rows)

FDS:
    ['Within_14_days', 'In_15_to_28_days', 'In_29_to_42_days', 'In_43_to_62_days', 'After_62_days']

31D:
    ['After_62_days', 'Within_31_days', 'In_32_to_38_days', 'In_39_to_48_days', 'In_49_to_62_days']

62D:
    ['After_62_days', 'Within_31_days', 'In_32_to_38_days', 'In_39_to_48_days', 'In_49_to_62_days', 'In_63_to_76_days', 'In_77_to_90_days', 'In_91_to_104_days', 'After_104_days']

Urgent Suspected Cancer referral:
    ['Within_14_days', 'In_15_to_16_days', 'In_17_to_21_days', 'In_22_to_28_days', 'After_28_days']

Urgent Breast Symptomatic referral:
    ['Within_14_days', 'In_15_to_16_days', 'In_17_to_21_days', 'In_22_to_28_days', 'After_28_days']



## Step 3 — Catalogue the dimension values

Before I build Bronze, I record every distinct value in my four dimension columns.
This is my data dictionary — it tells anyone reading the repo exactly what slices exist,
and it tells me what to expect when I aggregate in Silver and Gold.

**A naming caution I noted:** the band column names (e.g. `Within_31_days`, `After_62_days`)
are reused physically across standards, so the name does not always match the meaning for
a given standard. The reliable fields are `Total`, `Within`, `After`, `Performance`.
I treat the bands as supporting colour, not headline numbers.

In [20]:
# I catalogue every distinct value in my four dimension columns. This is my data dictionary.
dimension_cols = ['Basis', 'Standard_or_Item', 'Cancer_Type', 'Referral_Route_or_Stage', 'Treatment_Modality']

for col in dimension_cols:
    values = df_2425[col].dropna().unique()
    print(f"{col} — {len(values)} distinct values:")
    for v in sorted(values.astype(str)):
        print(f"    • {v}")
    print()

Basis — 2 distinct values:
    • Commissioner
    • Provider

Standard_or_Item — 5 distinct values:
    • 31D
    • 62D
    • FDS
    • Urgent Breast Symptomatic referral
    • Urgent Suspected Cancer referral

Cancer_Type — 32 distinct values:
    • ALL CANCERS
    • Breast
    • Exhibited (non-cancer) breast symptoms - cancer not initially suspected
    • Gynaecological
    • Haematological - Lymphoma
    • Haematological - Other (a)
    • Head & Neck
    • Lower Gastrointestinal
    • Lung
    • Missing or Invalid
    • Other (a)
    • Skin
    • Suspected acute leukaemia
    • Suspected brain/central nervous system tumours
    • Suspected breast cancer
    • Suspected cancer - non-specific symptoms
    • Suspected children's cancer
    • Suspected gynaecological cancer
    • Suspected haematological malignancies (excluding acute leukaemia)
    • Suspected head & neck cancer
    • Suspected lower gastrointestinal cancer
    • Suspected lung cancer
    • Suspected other cancer
    • 

In [22]:
# I confirm the trust coverage and time span, so my scope decision is on the record.
providers = df_2425[df_2425['Basis'] == 'Provider']
real_trusts = providers[providers['Org_Code'] != 'Total']['Org_Code'].nunique()

print("Provider trusts (excluding the England 'Total' row):", real_trusts)
print("Periods covered in this file:", df_2425['Period'].min(), "to", df_2425['Period'].max())
print("Number of months:", df_2425['Period'].nunique())
print()
print("My analysis scope for Silver/Gold: October 2023 onwards (post-framework, comparable).")
print("This 2024-25 file sits entirely inside that scope.")

Provider trusts (excluding the England 'Total' row): 172
Periods covered in this file: 2024-04-01 to 2025-03-01
Number of months: 12

My analysis scope for Silver/Gold: October 2023 onwards (post-framework, comparable).
This 2024-25 file sits entirely inside that scope.


## Exploration summary — what I know before building Bronze

**Scale:** 3 financial years, ~285k–325k rows each. 172 provider trusts. 12 months per file.

**The file holds 5 items:**
- `FDS` — Faster Diagnosis Standard (28 days)
- `31D` — 31-day decision-to-treat standard
- `62D` — 62-day referral-to-treatment standard
- `Urgent Suspected Cancer referral` (USC) — referral-to-first-appointment activity
- `Urgent Breast Symptomatic referral` (UBS) — referral-to-first-appointment activity

**Two bases:** Provider and Commissioner. I filter to **Provider** for analysis to avoid
double-counting — provider data is the most complete assessment of NHS performance.

**Reliable measure fields:** `Total`, `Within`, `After`, `Performance`. The waiting-time
band columns are reused physically across standards, so their names do not always match
their meaning per standard. I use bands as supporting colour only.

**Data quality issues I must fix in Silver:**
1. **Casing inconsistency** — `Urgent Suspected Cancer` vs `URGENT SUSPECTED CANCER`,
   `Screening` vs `NATIONAL SCREENING PROGRAMME`. I normalise these or I split real categories.
2. **Suppression** — small numbers (<5) appear as nulls. I preserve them as null, never coerce to 0.
3. **The 'Total' Org_Code** — the England aggregate row sits inside the data with `Org_Code='Total'`.
   I keep it for national figures but exclude it from any trust-level ranking.

**Scope:** Silver and Gold cover October 2023 onwards (post-framework, comparable across standards).

**My three articles map to three Gold cuts:**
1. The 62-Day Lottery → trust-level league table
2. It Depends What Cancer You Have → cancer-type breakdown
3. The Routes That Decide Your Wait → Consultant Upgrade vs Urgent Suspected Cancer